# Modelli di Raccomandazione: Baselines, CF, CB, Hybrid
**Corso:** Sistemi Intelligenti per Internet (SII)


In [ ]:
import sys
sys.path.append("..")

from sklearn.model_selection import train_test_split
from src.data_loader import download_and_extract_movielens, load_ratings, load_items, build_item_feature_matrix
from src.baselines import GlobalMeanBaseline, UserMeanBaseline, ItemMeanBaseline
from src.collaborative import CollaborativeFilteringModel
from src.content_based import ContentBasedRecommender
from src.hybrid import HybridRecommender
from src.evaluation import compute_rmse, compute_mae



## 1. Split Train / Test (80/20) e Training Modelli


In [ ]:
ml_dir = download_and_extract_movielens(data_dir="../data")
ratings_df = load_ratings(ml_dir)
items_df = load_items(ml_dir)

feature_matrix, item_id_to_idx, _ = build_item_feature_matrix(items_df, include_year=True)
train_df, test_df = train_test_split(ratings_df, test_size=0.2, random_state=42)

# Fit models
g_mean = GlobalMeanBaseline().fit(train_df)
u_mean = UserMeanBaseline().fit(train_df)
i_mean = ItemMeanBaseline().fit(train_df)

cf = CollaborativeFilteringModel(n_factors=50, n_epochs=20, random_state=42).fit(train_df)
cb = ContentBasedRecommender(feature_matrix, item_id_to_idx, rating_threshold=3.0).fit(train_df)
hybrid = HybridRecommender(cf, cb, alpha=0.5).fit(train_df)

print("Modelli addestrati con successo.")



## 2. Valutazione Singola Predizione


In [ ]:
y_true = test_df['rating'].values

models = {
    "Global Mean": g_mean.predict_batch(test_df),
    "User Mean": u_mean.predict_batch(test_df),
    "Item Mean": i_mean.predict_batch(test_df),
    "CF-only (SVD)": cf.predict_batch(test_df),
    "CB-only (Cosine)": cb.predict_batch(test_df),
    "Hybrid (alpha=0.5)": hybrid.predict_batch(test_df)
}

for name, preds in models.items():
    rmse = compute_rmse(y_true, preds)
    mae = compute_mae(y_true, preds)
    print(f"{name:20s} | RMSE: {rmse:.4f} | MAE: {mae:.4f}")

